# 102 - Orthogonalizing over Annular & Arbitrary Apertures

Zernikes are orthogonal on the unit circle.  However, it is common to encounter
systems which either have an obscured aperture, or a polygonal or more
complicated shape.

prysm has a utility function which can stretch an obscured
aperture to be a new unit circle, which allows for a Mahajan-like set of
annular Zernikes (or any polynomial orthogonal on the unit circle) to be made.
The benefit to this mode is that the exact mode shapes are independent of the
number of modes computed.  

prysm has a second utility, which uses QR factorization to compute a new set
of orthogonal modes, based on any set of input modes.  This process will deform
the original modes and because it produces orthogonality numerically, the exact
shape of a given mode depends on how large a basis is generated.  As a result,
when working in this style, you must communicate with collaborators and use
the same input modes (and # of modes) and in edge cases similar sampling
densities.  For example, someone using 32 pixels per hexagonal segment will not
get the same answer as someone using 512 samples per hexagonal segment.

This notebook will walk through both examples.  We'll start with some imports
and a working grid and zernike set:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from prysm.coordinates import (
    make_xy_grid,
    cart_to_polar,
    distort_annular_grid  # <---- focus of this notebook
)
from prysm.geometry import circle, regular_polygon

from prysm.polynomials import (
    zernike_nm_seq,
    fringe_to_nm,
    normalize_modes,
    orthogonalize_modes, # <---- focus of this notebook
)

# a 256x256 grid spanning the unit disk; r,t are polar; mask is True OUTSIDE
x, y = make_xy_grid(256, diameter=2)
r, t = cart_to_polar(x, y)
out = ~circle(1, r)        # boolean: True outside the unit circle (for blanking)

def show(arr, ax=None, blank=True, **kw):
    a = np.asarray(arr, dtype=float).copy()
    if blank:
        a[out] = np.nan
    ax = ax or plt.gca()
    return ax.imshow(a, cmap=kw.pop('cmap', 'RdBu'), **kw)


nms = [fringe_to_nm(j) for j in range(1, 37)]

## Annular Zernikes by grid distortion

The annular Zernikes are the circle Zernikes evaluated on a radius that stretches
the annulus $[\epsilon, 1]$ back onto the full $[0, 1]$.  `distort_annular_grid(r, eps)`
builds that remapped radius; feed it to `zernike_nm_seq` and re-`normalize_modes`
over the mask and you have analytically-orthogonal annular Zernikes.

In [ ]:
eps = 0.5                                  # inner/outer radius ratio
mask = circle(1, r) ^ circle(eps, r)       # XOR: the annulus
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(mask, cmap='gray'); ax.set(title=f'annular aperture, eps={eps}', xticks=[], yticks=[])

r_annular = distort_annular_grid(r, eps)
annular = zernike_nm_seq(nms, r_annular, t, norm=False)
annular = normalize_modes(annular, mask, to='std')

circ = zernike_nm_seq(nms, r, t, norm=False)   # plain circle Zernikes, for contrast

fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for j, name in zip((3, 4, 6, 8), ('Power', 'Astigmatism', 'Coma', 'Spherical')):
    col = (3, 4, 6, 8).index(j)
    for row, bas in enumerate((circ, annular)):
        m = bas[j].copy(); m[~mask] = np.nan
        axs[row, col].imshow(m, cmap='RdBu'); axs[row, col].set(xticks=[], yticks=[])
    axs[0, col].set_title(name)
axs[0, 0].set_ylabel('circle Zernikes'); axs[1, 0].set_ylabel('annular Zernikes')
fig.suptitle('the annular modes are re-shaped to stay orthogonal on the annulus')

The annular spherical mode has a different radial profile, pushed outward to be
orthogonal to power over the annulus.  The most striking changes tend to be
rotationally symmetric modes, since their central lump would be missing without
the transformation.

## Arbitrary apertures by QR

`distort_annular_grid` is specific to annuli.  For arbitrary apertures, you
will need to use `orthogonalize_modes(basis, mask)`:

In [ ]:
hexap = regular_polygon(6, 1, x, y, rotation=0) > 0    # a hexagonal aperture

circ = zernike_nm_seq(nms, r, t, norm=True)
hexed = orthogonalize_modes(circ, hexap)
hexed = normalize_modes(hexed, hexap, to='std')

fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for col, (j, name) in enumerate(zip((3, 6, 10, 28),
                                    ('Power', 'Coma', 'Trefoil', 'Quadrafoil'))):
    for row, bas in enumerate((circ, hexed)):
        m = bas[j].copy(); m[~hexap] = np.nan
        axs[row, col].imshow(m, cmap='RdBu', clim=(-3, 3))
        axs[row, col].set(xticks=[], yticks=[])
    axs[0, col].set_title(name)
axs[0, 0].set_ylabel('circle Zernikes'); axs[1, 0].set_ylabel('QR-orthogonalized')
fig.suptitle('QR re-orthogonalizes any basis over an arbitrary mask')

The QR modes are warped to respect the hexagon's corners - what it takes to stay
orthogonal there.  The same call works for any mask, which is why it is the
standard tool for segmented and obscured pupils (the Apertures, Segments & DMs
college uses it heavily).

## Wrap-up

This notebook walked through orthogonalization over domains that the original
polynomials are not orthogonal over.  The next notebook will walk through
all of the other polynomial bases, except for Forbes' Q-type polynomials
which have their own notebook due to their much h igher complexity.